# ELECTRO-VOLT — 04. Diagnóstico de Overfitting/Underfitting y Optimización
**Proyecto Integrador · Maestría en IA · UEES**

**Autores:** Jordan Vera Coello & Jorge Estrella Frere

En este notebook se analizan formalmente las curvas de aprendizaje y validación:
1. Diagnóstico de sesgo y varianza: Curvas de aprendizaje (`learning_curve` con `shuffle=True`).
2. Curvas de validación: Hiperparámetro `n_estimators` en Random Forest y `C` en SVM.
3. Evaluación de las 3 estrategias de optimización exploradas:
   - **Estrategia 1:** Regularización de árboles (`max_depth`, `min_samples_leaf`).
   - **Estrategia 2:** Tuning de hiperparámetros SVM ($C=10, 100$).
   - **Estrategia 3:** Fusión física a 5 clases (producción) vs Feature Engineering de 13 variables (documental).
4. Serialización oficial del modelo ganador en `models/best_model.pkl` y `models/modelo_fallos_rf.joblib`.


In [ ]:
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.ensemble import RandomForestClassifier

from src.data_processing import cargar_datos, mapear_clases, dividir_datos
from src.train import serializar_paquete_modelo
from src.utils import COLUMNAS, SEMILLA

sns.set_theme(style="whitegrid")
print("[OK] Módulos de optimización importados.")


## 1. Curva de aprendizaje
Analizamos si el Random Forest requiere más datos o sufre de memorización excesiva.
> **Nota crítica:** `learning_curve` utiliza `shuffle=True` debido a los bloques continuos del dataset.


In [ ]:
df = mapear_clases(cargar_datos(), modo="5_clases")
X, y = df[COLUMNAS], df["clase"]

tamanos, puntajes_ent, puntajes_val = learning_curve(
    RandomForestClassifier(n_estimators=100, random_state=SEMILLA, class_weight="balanced"),
    X, y, cv=5, scoring="f1_macro", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5),
    shuffle=True, random_state=SEMILLA
)

plt.figure(figsize=(8, 4.5))
plt.plot(tamanos, puntajes_ent.mean(axis=1), "o-", color="firebrick", label="Entrenamiento")
plt.plot(tamanos, puntajes_val.mean(axis=1), "s-", color="forestgreen", label="Validación Cruzada")
plt.title("Curva de Aprendizaje — Random Forest (5 Clases)", fontsize=13)
plt.xlabel("Muestras de Entrenamiento")
plt.ylabel("F1 Macro")
plt.ylim(0.95, 1.01)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 2. Curva de validación: Número de árboles (`n_estimators`)
Evaluamos el impacto del número de árboles en la estabilidad de la varianza.


In [ ]:
param_rango = [10, 25, 50, 100, 150]
punt_ent_v, punt_val_v = validation_curve(
    RandomForestClassifier(random_state=SEMILLA, class_weight="balanced"),
    X, y, param_name="n_estimators", param_range=param_rango,
    cv=3, scoring="f1_macro", n_jobs=-1
)

plt.figure(figsize=(8, 4))
plt.plot(param_rango, punt_ent_v.mean(axis=1), "o-", color="firebrick", label="Entrenamiento")
plt.plot(param_rango, punt_val_v.mean(axis=1), "s-", color="forestgreen", label="Validación")
plt.title("Curva de Validación: n_estimators en Random Forest", fontsize=13)
plt.xlabel("Número de Árboles (n_estimators)")
plt.ylabel("F1 Macro")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Síntesis de estrategias y serialización del mejor modelo
1. **E1 (Regularización):** Limitar profundidad en 6 clases empeoró el F1 sin resolver la ambigüedad física.
2. **E2 (Tuning SVM):** Con $C=10$ y StandardScaler se alcanza F1 de $0.979$, pero Random Forest es superior ($0.999$).
3. **E3 (Fusión Física):** La unificación a 5 clases es la solución matemáticamente óptima para producción ($Gap < 0.001$).
Guardamos el modelo oficial final en `models/best_model.pkl` y `models/modelo_fallos_rf.joblib`.


In [ ]:
X_ent, X_pru, y_ent, y_pru, _, _ = dividir_datos(df, test_size=0.20, random_state=SEMILLA)
rf_final = RandomForestClassifier(n_estimators=100, random_state=SEMILLA, class_weight="balanced")
rf_final.fit(X_ent, y_ent)

ruta_guardado = serializar_paquete_modelo(rf_final)
print(f"[OK] Modelo final oficial serializado exitosamente en: {ruta_guardado}")
